# Fine-tune DistilBERT on SMS messages

This notebook fine-tunes `distilbert-base-uncased` to classify SMS messages as
`benign` or `malicious`.

It uses the processed train/validation/test splits created in
`01_data_exploration.ipynb`, then saves the trained model and tokenizer to
`../models/distilbert-sms/`.

In [18]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "models" / "distilbert-sms"

train_df = pd.read_csv(DATA_DIR / "train.csv")
validation_df = pd.read_csv(DATA_DIR / "validation.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(train_df.shape)
print(validation_df.shape)
print(test_df.shape)

train_df.head()

(3901, 2)
(558, 2)
(1115, 2)


,message,label
0,Welcome to UK-mobile-date this msg is FREE giv...,malicious
1,Theyre doing it to lots of places. Only hospit...,benign
2,Are your freezing ? Are you home yet ? Will yo...,benign
3,They released vday shirts and when u put it on...,benign
4,Ok lor then we go tog lor...\r\n,benign


In [19]:
from sms_classifier.model import (
    dataframe_to_dataset,
    load_tokenizer,
    tokenize_messages,
)

train_dataset = dataframe_to_dataset(train_df)
validation_dataset = dataframe_to_dataset(validation_df)
test_dataset = dataframe_to_dataset(test_df)

tokenizer = load_tokenizer()

tokenized_train_dataset = train_dataset.map(
    lambda examples: tokenize_messages(examples, tokenizer), batched=True
)
tokenized_validation_dataset = validation_dataset.map(
    lambda examples: tokenize_messages(examples, tokenizer), batched=True
)
tokenized_test_dataset = test_dataset.map(
    lambda examples: tokenize_messages(examples, tokenizer), batched=True
)

print(tokenized_train_dataset)
print(tokenized_train_dataset[0].keys())

Map: 100%|██████████| 1115/1115 [00:00<00:00, 45702.09 examples/s]

Dataset({
    features: ['message', 'label', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3901
})
dict_keys(['message', 'label', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'])


In [20]:
import torch
from transformers import DataCollatorWithPadding, TrainingArguments

from sms_classifier.model import load_sequence_classifier

model = load_sequence_classifier()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model's weights to the selected device.
#
# If CUDA is available, this puts the model on the NVIDIA GPU, so training is
# much faster. Otherwise, the model stays on the CPU.
#
# During training, the batches also need to be on the same device as the model;
# Hugging Face Trainer handles that for us.
model.to(device)

print(f"Using device: {device}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print(
        "WARNING: CUDA is not available. Training will run on CPU and may be slow. "
        "For this project, prefer a small training configuration if using CPU."
    )

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6447.72it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Using device: cuda
NVIDIA GeForce RTX 4070 SUPER


The model-loading warning is expected. We start from `distilbert-base-uncased`,
which was pretrained for general language understanding, not SMS classification.
Hugging Face discards unused pretraining-head weights and initializes a new 
two-label classification head. 
Fine-tuning trains that new head for `benign` vs `malicious`.

In [21]:
training_arguments = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "models" / "training-checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

training_arguments

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval

In [22]:
import evaluate  # huggingface industry standard lib
import numpy as np
from transformers.trainer_utils import EvalPrediction

# Load metric implementations once.
# Hugging Face's `evaluate` library gives us reusable metric calculators.
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred: EvalPrediction) -> dict[str, float]:
    """Compute validation metrics during training.

    Hugging Face Trainer calls this function after evaluation.

    For this binary classifier:
    - label 0 means "benign"
    - label 1 means "malicious"
    """

    # `predictions` are raw model scores, also called logits.
    # Shape is roughly: (number_of_messages, number_of_labels)
    #
    # Example for one message:
    #   [-1.2, 2.4]
    #
    # That means:
    #   score for label 0 = -1.2
    #   score for label 1 =  2.4
    # The larger logit is the class the model currently favors.
    # These are raw scores, not probabilities
    logits = eval_pred.predictions

    # `label_ids` are the correct answers from the validation dataset.
    # Example:
    #   [0, 1, 0, 0, 1]
    true_labels = eval_pred.label_ids

    # Pick the label with the highest score for each message.
    # `argmax` returns the index of the biggest score, not the score itself.
    # Example:
    #   [-1.2, 2.4] -> 1 as output (predicted_labels variable)
    # because label 1 has the higher score.
    predicted_labels = np.argmax(logits, axis=-1)

    # Accuracy answers:
    # "What fraction of all predictions were correct?"
    # Compare model predictions against correct labels and calculate accuracy
    accuracy = accuracy_metric.compute(
        predictions=predicted_labels,
        references=true_labels,
    )["accuracy"]

    # F1 combines precision and recall into one score.
    #
    # For this project, we care most about the "malicious" class:
    # - precision: when the model says "malicious", how often it is right
    # - recall: of all truly malicious messages, how many the model catches
    #
    # `average="binary"` computes F1 for one selected class.
    # `pos_label=1` selects label 1, which is "malicious" in this project.
    # In this project, label 1 = "malicious"
    f1 = f1_metric.compute(
        predictions=predicted_labels,
        references=true_labels,
        average="binary",
        pos_label=1,
    )["f1"]

    return {
        "accuracy": float(accuracy),
        "f1": float(f1),
    }

In [23]:
from transformers import Trainer

# The tokenizer creates token lists of different lengths because SMS messages
# are different lengths.
#
# A training batch must be a rectangular tensor, so examples in the same batch
# need to be padded to the same length.
#
# DataCollatorWithPadding does that padding dynamically per batch instead of
# padding every message to MAX_LENGTH ahead of time.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

columns_to_remove = ["message", "label"]

train_features = tokenized_train_dataset.remove_columns(columns_to_remove)
validation_features = tokenized_validation_dataset.remove_columns(columns_to_remove)
#test_features = tokenized_test_dataset.remove_columns(columns_to_remove)

print(train_features.column_names)
print(train_features[0])

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_features,
    eval_dataset=validation_features,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print(trainer.args.output_dir)
print(len(trainer.train_dataset))
print(len(trainer.eval_dataset))

['labels', 'input_ids', 'token_type_ids', 'attention_mask']
{'labels': 1, 'input_ids': [101, 6160, 2000, 2866, 1011, 4684, 1011, 3058, 2023, 5796, 2290, 2003, 2489, 3228, 2017, 2489, 4214, 2000, 5511, 2581, 16147, 2620, 23499, 2620, 19481, 1012, 2925, 11460, 2015, 14843, 2012, 5018, 2361, 3679, 1012, 2000, 17542, 4604, 1000, 2175, 2644, 1000, 2000, 6486, 12521, 2509, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
C:\Users\giloz\dev\sms-malicious-classifier\models\training-checkpoints
3901
558


## Train the model

This step fine-tunes the pretrained DistilBERT model on the SMS training split.
The validation split is evaluated after each epoch. The configuration is kept
small for this learning project: two epochs and moderate batch sizes.

In [24]:
train_result = trainer.train()
train_result

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.070212,0.058010,0.987455,0.951724
2,0.023470,0.059383,0.987455,0.953020


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.11it/s]


TrainOutput(global_step=488, training_loss=0.055533556847787297, metrics={'train_runtime': 151.7759, 'train_samples_per_second': 51.405, 'train_steps_per_second': 3.215, 'total_flos': 123170878014384.0, 'train_loss': 0.055533556847787297, 'epoch': 2.0})

In [25]:
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print(f"Saved model and tokenizer to: {MODEL_OUTPUT_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.54it/s]

Saved model and tokenizer to: C:\Users\giloz\dev\sms-malicious-classifier\models\distilbert-sms


In [26]:
print("Saved model files:")
for path in sorted(MODEL_OUTPUT_DIR.iterdir()):
    print(path.name)

Saved model files:
config.json
model.safetensors
tokenizer.json
tokenizer_config.json
training_args.bin
